# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Vasavijoshi/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

### Finding 1 — Growing pages are younger

The report compares growing and declining pages and observes that growing pages are younger on average (185 days versus 228 days), while average word count is nearly the same. My methodology question is: how exactly is the growing/declining label constructed, and does the comparison support only an association between page age and traffic direction, rather than a causal claim that older pages decline because they are old? I would also want to confirm that the two groups were comparable on other important factors before interpreting age as the main explanation.

### Finding 2 — Refreshing pages shows positive lift

The report states that 7 of 9 strata show statistically significant refresh lift and reports a median effect of +588 impressions for refreshed versus stale pages among pages older than 180 days. My methodology question is: how were pages assigned to the refreshed and stale groups, and does the validation design rule out important pre-existing differences between those groups? If refresh assignment was observational rather than randomized, I would treat the result as measured evidence of an association or directional effect rather than proof that refreshing itself caused the lift.

### Why I am asking these questions

These are constructive validation questions, not claims that the findings are wrong. They identify where label construction, group comparability, and validation design matter when moving from an observed result to a stronger causal interpretation.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

### Validation comparison

The earlier comparison uses a random row split, which can place pages from the same client in both training and test data. I treat this as an optimistic reference rather than a trustworthy estimate for unseen clients. I then re-run the same model with an 80/20 client-grouped split, keeping every client's pages in only one side of the split. The grouped result is the more honest estimate for this lane.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ML-09 Section 2 — Before/after validation audit

import os
import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.pipeline import Pipeline

# Load data
repo_path = "/content/flyrank-ml-internship"

if not os.path.exists(repo_path):
    !git clone https://github.com/Vasavijoshi/flyrank-ml-internship.git

df = pd.read_csv(
    f"{repo_path}/data/raw/content_refresh_anonymized.csv"
)

# Outcome: used only as the target, never as a feature
df["is_declining_label"] = (
    df["trend_direction"] == "down"
).astype(int)

feature_cols = [
    "days_since_last_update",
    "impressions_90d",
    "clicks_90d",
    "sessions_90d",
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate",
    "content_age_days",
    "word_count",
    "search_volume",
    "cpc"
]

target_col = "is_declining_label"

# ---------------------------------------------------------
# Helper: build and evaluate the same Random Forest
# ---------------------------------------------------------

def precision_at_k(y_true, scores, k):
    order = np.argsort(-scores)[:k]
    return np.mean(np.asarray(y_true)[order])


def train_and_score(train_df, test_df):
    X_train = train_df[feature_cols]
    X_test = test_df[feature_cols]

    y_train = train_df[target_col]
    y_test = test_df[target_col]

    model = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("rf", RandomForestClassifier(
            n_estimators=300,
            random_state=42,
            n_jobs=-1,
            min_samples_leaf=5
        ))
    ])

    model.fit(X_train, y_train)

    probabilities = model.predict_proba(X_test)[:, 1]

    return {
        "Precision@10": precision_at_k(y_test, probabilities, 10),
        "Precision@50": precision_at_k(y_test, probabilities, 50),
        "Precision@100": precision_at_k(y_test, probabilities, 100),
        "test_rows": len(test_df),
        "test_clients": test_df["client_id"].nunique()
    }


# ---------------------------------------------------------
# BEFORE: random row split
# ---------------------------------------------------------

before_train, before_test = train_test_split(
    df,
    test_size=0.20,
    random_state=42,
    stratify=df[target_col]
)

before = train_and_score(before_train, before_test)


# ---------------------------------------------------------
# AFTER: grouped client split
# ---------------------------------------------------------

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(
        df,
        y=df[target_col],
        groups=df["client_id"]
    )
)

after_train = df.iloc[train_idx].copy()
after_test = df.iloc[test_idx].copy()

after = train_and_score(after_train, after_test)


# ---------------------------------------------------------
# Compare
# ---------------------------------------------------------

comparison = pd.DataFrame([
    {
        "validation": "Before — random row split",
        "Precision@10": before["Precision@10"],
        "Precision@50": before["Precision@50"],
        "Precision@100": before["Precision@100"],
        "test_rows": before["test_rows"],
        "test_clients": before["test_clients"]
    },
    {
        "validation": "After — grouped client split",
        "Precision@10": after["Precision@10"],
        "Precision@50": after["Precision@50"],
        "Precision@100": after["Precision@100"],
        "test_rows": after["test_rows"],
        "test_clients": after["test_clients"]
    }
])

display(comparison.round(4))

# Explicit leakage check for the grouped split
client_overlap = (
    set(after_train["client_id"])
    & set(after_test["client_id"])
)

print("Grouped split client overlap:", len(client_overlap))
print("Grouped training clients:", after_train["client_id"].nunique())
print("Grouped test clients:", after_test["client_id"].nunique())

,validation,Precision@10,Precision@50,Precision@100,test_rows,test_clients
0,Before — random row split,1.0,0.96,0.94,6000,31
1,After — grouped client split,0.5,0.64,0.61,6163,7


Grouped split client overlap: 0
Grouped training clients: 25
Grouped test clients: 7


### Before/after interpretation

The random row split produced much higher Precision@K values, including 1.00 at Precision@10. However, this split allowed pages from the same clients to appear in both training and test data, so it is an optimistic reference rather than an honest estimate for unseen clients. After grouping by client, Precision@10 fell to 0.50, Precision@50 to 0.64, and Precision@100 to 0.61. The grouped split had zero client overlap and is therefore the more credible validation result for this lane. This audit reinforces that the Week-5 grouped result, rather than the random-split result, should be used for decision-support claims.

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

### Leakage audit

The final model uses 12 features that are available before the outcome is evaluated. I explicitly exclude `trend_direction` and `trend_pct` because the target is constructed from `trend_direction`, and `trend_pct` directly describes the outcome trend. I also exclude identifiers and other fields that are not appropriate predictive inputs. The checks below confirm that no forbidden outcome-derived fields are present in the final feature set.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ML-09 Section 3 — Leakage audit

# Final feature set used by the Random Forest
final_features = [
    "days_since_last_update",
    "impressions_90d",
    "clicks_90d",
    "sessions_90d",
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate",
    "content_age_days",
    "word_count",
    "search_volume",
    "cpc"
]

# Fields that must NOT be model features
forbidden_features = [
    "trend_direction",
    "trend_pct",
    "is_declining_label",
    "content_id",
    "client_id"
]

# 1. Check for forbidden fields
forbidden_used = sorted(
    set(final_features) & set(forbidden_features)
)

print("Final feature count:", len(final_features))
print("Forbidden fields used:", forbidden_used)

# 2. Check that the target is not in the feature set
print(
    "Target included as feature:",
    target_col in final_features
)

# 3. Check that every final feature exists in the dataset
missing_features = [
    feature for feature in final_features
    if feature not in df.columns
]

print("Missing final features:", missing_features)

# 4. Check for obvious outcome-derived feature names
suspicious_names = [
    feature for feature in final_features
    if any(
        term in feature.lower()
        for term in ["trend", "declin", "label", "outcome", "future"]
    )
]

print("Outcome-like feature names:", suspicious_names)

# 5. Final audit verdict
leakage_audit_pass = (
    len(forbidden_used) == 0
    and target_col not in final_features
    and len(missing_features) == 0
    and len(suspicious_names) == 0
)

print("\nLeakage audit:", "PASS" if leakage_audit_pass else "REVIEW")

Final feature count: 12
Forbidden fields used: []
Target included as feature: False
Missing final features: []
Outcome-like feature names: []

Leakage audit: PASS


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

### Claim rewrite

**Original stronger claim:**  
“The Random Forest significantly improves the identification of declining pages compared with the Week-4 baseline.”

**Safer claim:**  
“On the client-grouped holdout used for decision-support evaluation, the Random Forest measured higher Precision@K than the Week-4 rule-based baseline: 0.50 versus 0.30 at K=10, 0.64 versus 0.30 at K=50, and 0.61 versus 0.30 at K=100. These are observed results on this dataset and split; they support the model as a directional ranking aid, not proof that the model will generalize to every client or that the ranked pages will decline because of the recommended action.”

**Validation lesson:**  
“The random row split produced substantially higher Precision@K, but it allowed pages from the same clients in both train and test. The grouped-client result is therefore the more credible estimate for unseen-client decision support.”

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ML-09 Section 4 — Claim rewrite evidence

claim_evidence = pd.DataFrame([
    {
        "metric": "Precision@10",
        "baseline": 0.30,
        "random_row_split": before["Precision@10"],
        "grouped_client_split": after["Precision@10"]
    },
    {
        "metric": "Precision@50",
        "baseline": 0.30,
        "random_row_split": before["Precision@50"],
        "grouped_client_split": after["Precision@50"]
    },
    {
        "metric": "Precision@100",
        "baseline": 0.30,
        "random_row_split": before["Precision@100"],
        "grouped_client_split": after["Precision@100"]
    }
])

display(claim_evidence.round(4))

print(
    "Safe claim basis: grouped-client validation, "
    "observed Precision@K, directional decision support."
)

,metric,baseline,random_row_split,grouped_client_split
0,Precision@10,0.3,1.00,0.50
1,Precision@50,0.3,0.96,0.64
2,Precision@100,0.3,0.94,0.61


Safe claim basis: grouped-client validation, observed Precision@K, directional decision support.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.